In [ ]:
#dette er den koden som fungerte best for meg for å fjerne støy
#settup av data
import hyperspy.api as hs
import pyxem as pxm
import matplotlib.pyplot as plt
from skimage import data
from skimage.filters import threshold_otsu
s = hs.load("006.zspy", lazy=True)
s_bs = s.get_direct_beam_position(method="center_of_mass")
s_bs.compute()
s_bs_lp = s_bs.get_linear_plane()
s_bs_corr = s_bs-s_bs_lp
s_sum = s.sum(axis=(-1, -2))
s_sum.compute()


s_bs.inav[10.:20., 10.:20.].plot()

s_bs_corr.isig[0].inav[10.:20., 10.:20.].plot()

s_bs_corr.get_magnitude_signal().isig[10.:20., 10.:20.].plot()
#finner maske 
thresh = threshold_otsu(s_sum.data)
mask = s_sum > thresh
#filtrerer vekk støy fra masken
#PCA med maskering (Anbefalt for diffrasjonsdata)
# 1. Lag en grov maske basert på intensitet (sann der strukturen er)

# 2. Kjør PCA kun på områdene med struktur
s_bs.decomposition(output_dimension=5, navigation_mask=mask)

# 3. Rekonstruer hele signalet (støyen i bakgrunnen blir undertrykt)
s_bs_denoised = s_bs.get_decomposition_model(components=3)

s_bs_denoised.inav[14.:18., 10.:20.].plot()
s_bs.inav[14.:18., 10.:20.].plot()
#decomposition funket veldig bra

print ("test")
